# Ноутбук 01: Filter methods (VarianceThreshold, Correlation, Mutual Info, ANOVA)
Выполните все ячейки, затем обязательные самостоятельные задания.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, f_classif
from lab_utils import (
    load_dataset, split_xy, train_test_split_stratified, build_preprocessor,
    transform_with_names, append_ranking_rows, build_shortlist, to_dense
)
%matplotlib inline
sns.set_style('whitegrid')

In [2]:
# Загрузка медицинского датасета
df_med = load_dataset('../data/medical.csv')
X_med, y_med = split_xy(df_med)
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split_stratified(X_med, y_med)

df_fin = load_dataset('../data/finance.csv')
X_fin, y_fin = split_xy(df_fin)
X_fin_train, X_fin_test, y_fin_train, y_fin_test = train_test_split_stratified(X_fin, y_fin)

print(f"Medical: train {X_med_train.shape}, test {X_med_test.shape}")
print(f"Finance: train {X_fin_train.shape}, test {X_fin_test.shape}")

Medical: train (12, 12), test (3, 12)
Finance: train (12, 11), test (3, 11)


In [3]:
# Препроцессинг
preprocessor_med = build_preprocessor(X_med_train)
X_med_train_t, X_med_test_t, feat_names_med = transform_with_names(preprocessor_med, X_med_train, X_med_test)

preprocessor_fin = build_preprocessor(X_fin_train)
X_fin_train_t, X_fin_test_t, feat_names_fin = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

In [4]:
# VarianceThreshold
rows = []
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_med_train_t)
var_scores = selector.variances_
append_ranking_rows(rows, 'medical', 'VarianceThreshold', feat_names_med, var_scores)

selector.fit(X_fin_train_t)
var_scores_fin = selector.variances_
append_ranking_rows(rows, 'finance', 'VarianceThreshold', feat_names_fin, var_scores_fin)

ranking_df = pd.DataFrame(rows)

In [5]:
# Корреляция с таргетом
for ds_name, X_t, y_t, fnames in [('medical', X_med_train_t, y_med_train, feat_names_med),
                                   ('finance', X_fin_train_t, y_fin_train, feat_names_fin)]:
    corr_scores = []
    for i, col in enumerate(fnames):
        x_col = to_dense(X_t[:, i]) if hasattr(X_t, 'tocsc') else X_t[:, i]
        corr = np.corrcoef(x_col, y_t)[0, 1]
        corr_scores.append(abs(corr) if not np.isnan(corr) else 0.0)
    append_ranking_rows(rows, ds_name, 'Correlation', fnames, np.array(corr_scores))

ranking_df = pd.DataFrame(rows)

In [6]:
# Mutual Information
mi_med = mutual_info_classif(X_med_train_t, y_med_train, random_state=42)
append_ranking_rows(rows, 'medical', 'MutualInfo', feat_names_med, mi_med)

mi_fin = mutual_info_classif(X_fin_train_t, y_fin_train, random_state=42)
append_ranking_rows(rows, 'finance', 'MutualInfo', feat_names_fin, mi_fin)
ranking_df = pd.DataFrame(rows)

In [7]:
# ANOVA F-test
f_med, p_med = f_classif(X_med_train_t, y_med_train)
append_ranking_rows(rows, 'medical', 'ANOVA', feat_names_med, f_med)

f_fin, p_fin = f_classif(X_fin_train_t, y_fin_train)
append_ranking_rows(rows, 'finance', 'ANOVA', feat_names_fin, f_fin)

ranking_df = pd.DataFrame(rows)
ranking_df.to_csv('../outputs/feature_ranking.csv', index=False)
print("Saved feature_ranking.csv")

Saved feature_ranking.csv


C:\Users\User\Desktop\01-feature-importance-and-selection\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [10] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\User\Desktop\01-feature-importance-and-selection\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


In [8]:
# Shortlist
filter_methods = ['VarianceThreshold', 'Correlation', 'MutualInfo', 'ANOVA']
shortlist_med = build_shortlist(ranking_df, 'medical', filter_methods, top_n=12)
shortlist_fin = build_shortlist(ranking_df, 'finance', filter_methods, top_n=12)
print("Medical shortlist:", shortlist_med)
print("Finance shortlist:", shortlist_fin)

Medical shortlist: ['num__cholesterol', 'num__glucose', 'num__exercise', 'num__systolic_bp', 'num__weight', 'num__family_history', 'num__diastolic_bp', 'num__smoking', 'num__alcohol', 'num__age', 'num__height', 'num__gender']
Finance shortlist: ['num__previous_default', 'num__debt_ratio', 'num__credit_score', 'num__interest_rate', 'cat__marital_status_married', 'num__dependents', 'num__income', 'num__loan_amount', 'cat__home_ownership_rent', 'num__employment_years', 'num__age', 'cat__home_ownership_own']


## Обязательное самостоятельное задание 1: устойчивость filter-ранжирования

In [9]:
variance_thresholds = [0.0, 0.001, 0.01, 0.05, 0.1]
top_n_list = [5, 10, 15, 20]
baseline_shortlist_med = set(shortlist_med)
baseline_shortlist_fin = set(shortlist_fin)

stability_rows = []
for dataset, X_train, y_train, feat_names, baseline in [
    ('medical', X_med_train_t, y_med_train, feat_names_med, baseline_shortlist_med),
    ('finance', X_fin_train_t, y_fin_train, feat_names_fin, baseline_shortlist_fin)
]:
    for vt in variance_thresholds:
        sel = VarianceThreshold(threshold=vt)
        sel.fit(X_train)
        var_scores = sel.variances_
        temp_rows = []
        append_ranking_rows(temp_rows, dataset, 'temp', feat_names, var_scores)
        temp_df = pd.DataFrame(temp_rows)
        for top_n in top_n_list:
            top_feats = temp_df[temp_df['method']=='temp'].sort_values('score', ascending=False).head(top_n)['feature'].tolist()
            overlap = len(set(top_feats) & baseline) / len(baseline) if baseline else 0
            stability_rows.append({
                'dataset': dataset,
                'variance_threshold': vt,
                'top_n': top_n,
                'shortlist_json': json.dumps(top_feats),
                'overlap_with_baseline': overlap
            })

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv('../outputs/filter_stability_grid.csv', index=False)
print("Saved filter_stability_grid.csv")

Saved filter_stability_grid.csv


In [ ]:
## Самостоятельное изучение по ходу работы (ноутбук 01 – Filter methods)

### Что изучено:
- **VarianceThreshold** – удаляет признаки с низкой дисперсией (почти константные). Полезен для первичной фильтрации, но не учитывает связь с таргетом.
- **Корреляция Пирсона** – измеряет линейную зависимость. На наших данных топ-признаки совпали с ANOVA, что говорит о примерно линейном характере связей.
- **Mutual Information (MI)** – улавливает нелинейные и сложные зависимости. В медицинском датасете MI выделила `smoking` и `exercise`, которые не попали в топ корреляции – значит, их влияние не строго линейное.
- **ANOVA F-test** – похож на корреляцию, предполагает линейность и нормальность. Признаки-константы вызвали warning, что нормально.

### Сравнение методов:
- Фильтры очень быстрые, работают с любыми моделями. 
- VarianceThreshold полезен для удаления мусора, но не ранжирует по важности.
- MI дала наиболее «разумный» с клинической точки зрения топ (курение, возраст, ИМТ, физическая активность).

### Источники:
- sklearn документация: [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html)
- [Mutual Information vs Correlation](https://stats.stackexchange.com/questions/81659/mutual-information-versus-correlation)

### Термины из глоссария, использованные в этом разделе:
- Дисперсия (VarianceThreshold)
- Взаимная информация (Mutual Information)
- F-статистика (ANOVA F-test)